# Large-Scale Hybrid Music Recommender — Data Pipeline
Generates: `data/large_hybrid/{song_characteristics,lyric_embeddings,interactions,user_features}.csv`

**Run cells top-to-bottom. All paths assume this notebook lives in the project root.**

## Cell 1 — Install / verify dependencies

In [3]:
# Cell 1 — Install / verify dependencies
import subprocess, sys

required = [
    'pandas', 'numpy', 'scikit-learn',
    'sentence-transformers', 'tqdm'
]
for pkg in required:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ All dependencies ready')

✅ All dependencies ready


## Cell 2 — Imports & global config

In [5]:
# Cell 2 — Imports & global config
import os, gc, warnings, random, json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
# songs.csv  = your full ~400k-row dataset
# songs_sample.csv = 5k representative sample (used when full file absent)
SONGS_PATH        = Path('/Users/suvratsmacbookair/Downloads/archive-3/songs.csv')          # full dataset
SONGS_SAMPLE_PATH = Path('/Users/suvratsmacbookair/Downloads/songs_sample.csv')   # fallback / dev
SPOTIFY_PATH      = Path('/Users/suvratsmacbookair/MyLocalFiles/LyricLens/spotify_songs.csv')
OUT_DIR           = Path('data/large_hybrid')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Scale knobs ────────────────────────────────────────────────────────────
TARGET_SONGS   = 30_000   # subset size (20k–50k). Reduce if RAM is tight.
N_USERS        = 350      # simulated users
N_POSITIVES    = 20       # positives per user
N_NEGATIVES    = 20       # negatives per positive
HARD_NEG_RATIO = 0.40     # fraction of negatives that are 'hard'
EMBED_BATCH    = 128      # SentenceTransformer batch size (safe on M2)
MIN_LYRIC_LEN  = 60       # characters — songs with shorter lyrics are dropped

AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

print(f'Output dir : {OUT_DIR.resolve()}')
print(f'Target songs: {TARGET_SONGS:,}')
print(f'Users       : {N_USERS}')

Output dir : /Users/suvratsmacbookair/MyLocalFiles/LyricLens/notebooks/data/large_hybrid
Target songs: 30,000
Users       : 350


## Cell 3 — Load & clean songs.csv (chunked, memory-safe)

In [7]:
# Cell 3 — Load & clean songs.csv (chunked, memory-safe)
#
# Strategy:
#   • If songs.csv exists  → chunked read, reservoir-sample TARGET_SONGS rows
#   • Otherwise            → use songs_sample.csv (dev / smoke-test mode)
#
# Columns kept from songs.csv:
#   id, name, artists, album_name, genre, year, popularity,
#   total_artist_followers, avg_artist_popularity, lyrics  +  AUDIO_FEATURES

KEEP_COLS = (['id', 'name', 'artists', 'album_name', 'genre', 'year',
              'popularity', 'total_artist_followers', 'avg_artist_popularity',
              'lyrics'] + AUDIO_FEATURES)

def _clean_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Drop nulls, short lyrics, deduplicate within a chunk."""
    chunk = chunk.dropna(subset=['id', 'lyrics', 'genre'])
    chunk = chunk[chunk['lyrics'].str.len() >= MIN_LYRIC_LEN]
    chunk = chunk.drop_duplicates(subset=['id'])
    # normalise genre capitalisation
    chunk['genre'] = chunk['genre'].str.strip().str.title()
    return chunk


def reservoir_sample_csv(path: Path, n: int, chunksize: int = 10_000) -> pd.DataFrame:
    """Memory-bounded reservoir sampling from a large CSV."""
    reservoir = []
    total_seen = 0
    rng = np.random.default_rng(42)

    for chunk in tqdm(pd.read_csv(path, usecols=KEEP_COLS,
                                  chunksize=chunksize, low_memory=False),
                      desc='Sampling songs.csv'):
        chunk = _clean_chunk(chunk)
        rows  = chunk.to_dict('records')
        for row in rows:
            total_seen += 1
            if len(reservoir) < n:
                reservoir.append(row)
            else:
                j = int(rng.integers(0, total_seen))
                if j < n:
                    reservoir[j] = row

    print(f'Total rows scanned: {total_seen:,} → sampled {len(reservoir):,}')
    return pd.DataFrame(reservoir)


if SONGS_PATH.exists():
    print(f'Found {SONGS_PATH} — reservoir-sampling {TARGET_SONGS:,} songs …')
    songs_df = reservoir_sample_csv(SONGS_PATH, TARGET_SONGS)
else:
    print(f'⚠️  songs.csv not found — falling back to {SONGS_SAMPLE_PATH}')
    songs_df = pd.read_csv(SONGS_SAMPLE_PATH, usecols=KEEP_COLS, low_memory=False)
    songs_df = _clean_chunk(songs_df)
    print(f'Loaded {len(songs_df):,} rows from sample')

songs_df = songs_df.drop_duplicates(subset=['id']).reset_index(drop=True)

# Ensure numeric types
for col in AUDIO_FEATURES + ['popularity', 'year', 'total_artist_followers', 'avg_artist_popularity']:
    if col in songs_df.columns:
        songs_df[col] = pd.to_numeric(songs_df[col], errors='coerce')

songs_df['popularity']               = songs_df['popularity'].fillna(0)
songs_df['total_artist_followers']   = songs_df['total_artist_followers'].fillna(0)
songs_df['avg_artist_popularity']    = songs_df['avg_artist_popularity'].fillna(0)
songs_df[AUDIO_FEATURES]             = songs_df[AUDIO_FEATURES].fillna(songs_df[AUDIO_FEATURES].median())

print(f'\n✅ songs_df shape : {songs_df.shape}')
print(f'Genre distribution:\n{songs_df["genre"].value_counts().head(12)}')

Found /Users/suvratsmacbookair/Downloads/archive-3/songs.csv — reservoir-sampling 30,000 songs …


Sampling songs.csv: 0it [00:00, ?it/s]

Total rows scanned: 550,004 → sampled 30,000

✅ songs_df shape : (30000, 19)
Genre distribution:
genre
Rock          10635
Pop            4048
Electronic     3804
Folk           2758
Country        2571
Hip-Hop        2305
R&B            1459
Jazz            899
Blues           857
Classical       664
Name: count, dtype: int64


## Cell 4 — Load & merge spotify_songs.csv

In [17]:
# Cell 4 — Load & merge spotify_songs.csv

import os
import pandas as pd

# ---------------------------------------------------------
# 1. Locate spotify_songs.csv automatically
# ---------------------------------------------------------

PROJECT_DIR = "/Users/suvratsmacbookair/MyLocalFiles/LyricLens"

# Possible locations
possible_paths = [
    os.path.join(PROJECT_DIR, "spotify_songs.csv"),
    os.path.join(PROJECT_DIR, "data", "spotify_songs.csv"),
    os.path.join(PROJECT_DIR, "datasets", "spotify_songs.csv"),
]

SPOTIFY_PATH = next(
    (path for path in possible_paths if os.path.isfile(path)),
    None
)

# If not found in the common locations, search the whole project folder
if SPOTIFY_PATH is None:
    for root, dirs, files in os.walk(PROJECT_DIR):
        if "spotify_songs.csv" in files:
            SPOTIFY_PATH = os.path.join(root, "spotify_songs.csv")
            break

if SPOTIFY_PATH is None:
    raise FileNotFoundError(
        f"❌ spotify_songs.csv was not found anywhere inside:\n{PROJECT_DIR}\n\n"
        "Please put spotify_songs.csv inside the LyricLens folder "
        "or one of its subfolders."
    )

print(f"✅ Found spotify_songs.csv at:\n{SPOTIFY_PATH}")


# ---------------------------------------------------------
# 2. Load Spotify dataset
# ---------------------------------------------------------

spotify_df = pd.read_csv(
    SPOTIFY_PATH,
    low_memory=False
)

print(f"✅ Spotify dataset loaded: {spotify_df.shape}")


# ---------------------------------------------------------
# 3. Keep English songs only
# ---------------------------------------------------------

if "language" in spotify_df.columns:
    spotify_df = spotify_df[
        spotify_df["language"].astype(str).str.lower().eq("en")
    ].copy()
else:
    print("⚠️ 'language' column not found. Skipping English-only filter.")


# ---------------------------------------------------------
# 4. Rename columns
# ---------------------------------------------------------

spotify_df = spotify_df.rename(columns={
    "track_id": "id",
    "track_name": "name",
    "track_popularity": "spotify_popularity",
})


# ---------------------------------------------------------
# 5. Check required columns
# ---------------------------------------------------------

required_columns = [
    "id",
    "spotify_popularity",
    "playlist_genre",
    "playlist_subgenre",
    "language"
]

missing_columns = [
    col for col in required_columns
    if col not in spotify_df.columns
]

if missing_columns:
    raise ValueError(
        f"❌ Missing columns in spotify_songs.csv: {missing_columns}\n\n"
        f"Available columns are:\n{list(spotify_df.columns)}"
    )


# ---------------------------------------------------------
# 6. Create slim Spotify dataframe
# ---------------------------------------------------------

SPOTIFY_KEEP = [
    "id",
    "spotify_popularity",
    "playlist_genre",
    "playlist_subgenre",
    "language"
]

spotify_slim = (
    spotify_df[SPOTIFY_KEEP]
    .drop_duplicates(subset=["id"])
    .copy()
)


# ---------------------------------------------------------
# 7. Build genre → subgenre lookup
# ---------------------------------------------------------

genre_subgenre_map = (
    spotify_df
    .groupby("playlist_genre")["playlist_subgenre"]
    .agg(
        lambda x: (
            x.dropna().mode().iloc[0]
            if not x.dropna().empty
            else "unknown"
        )
    )
    .to_dict()
)


# ---------------------------------------------------------
# 8. Normalised genre map
# ---------------------------------------------------------

genre_norm_map = {
    "rock": "Rock",
    "pop": "Pop",
    "r&b": "R&B",
    "rap": "Hip-Hop",
    "hip hop": "Hip-Hop",
    "hip-hop": "Hip-Hop",
    "latin": "Latin",
    "edm": "Electronic",
    "electronic": "Electronic",
    "country": "Country",
}


# ---------------------------------------------------------
# 9. Direct ID merge
# ---------------------------------------------------------

if "id" not in songs_df.columns:
    raise ValueError(
        "❌ songs_df does not contain an 'id' column."
    )

merged = songs_df.merge(
    spotify_slim,
    on="id",
    how="left"
)


# ---------------------------------------------------------
# 10. Fill missing subgenres
# ---------------------------------------------------------

def fill_subgenre(row):

    # Already matched from Spotify
    if pd.notna(row.get("playlist_subgenre")):
        return row["playlist_subgenre"]

    # Try to infer from existing genre
    genre = row.get("genre")

    if pd.isna(genre):
        return "unknown"

    genre = str(genre).lower().strip()

    for key, spotify_genre in genre_norm_map.items():

        if key in genre:

            # Spotify's actual genre names may have
            # different capitalization
            for spotify_key, subgenre in genre_subgenre_map.items():

                if str(spotify_key).lower() == spotify_genre.lower():
                    return subgenre

            return "unknown"

    return "unknown"


merged["playlist_subgenre"] = merged.apply(
    fill_subgenre,
    axis=1
)


# ---------------------------------------------------------
# 11. Fill missing popularity
# ---------------------------------------------------------

merged["spotify_popularity"] = pd.to_numeric(
    merged["spotify_popularity"],
    errors="coerce"
)

merged["popularity"] = pd.to_numeric(
    merged["popularity"],
    errors="coerce"
)

merged["spotify_popularity"] = (
    merged["spotify_popularity"]
    .fillna(merged["popularity"])
)


# ---------------------------------------------------------
# 12. Fill missing language
# ---------------------------------------------------------

merged["language"] = (
    merged["language"]
    .fillna("en")
)


# ---------------------------------------------------------
# 13. Effective popularity
# ---------------------------------------------------------

merged["effective_popularity"] = (
    0.6 * merged["popularity"].fillna(0)
    + 0.4 * merged["spotify_popularity"].fillna(0)
).clip(0, 100)


# ---------------------------------------------------------
# 14. Final diagnostics
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("✅ MERGE COMPLETE")
print("=" * 60)

print(f"Original songs_df shape : {songs_df.shape}")
print(f"Spotify dataset shape   : {spotify_df.shape}")
print(f"Merged shape            : {merged.shape}")

subgenre_coverage = (
    (merged["playlist_subgenre"] != "unknown").mean() * 100
)

print(f"Subgenre coverage        : {subgenre_coverage:.1f}%")

spotify_match = (
    merged["spotify_popularity"].notna().mean() * 100
)

print(f"Spotify match coverage  : {spotify_match:.1f}%")

print("=" * 60)

✅ Found spotify_songs.csv at:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/B/spotify_songs.csv
✅ Spotify dataset loaded: (18454, 25)

✅ MERGE COMPLETE
Original songs_df shape : (30000, 19)
Spotify dataset shape   : (15405, 25)
Merged shape            : (30000, 24)
Subgenre coverage        : 54.6%
Spotify match coverage  : 100.0%


## Cell 5 — Build song_characteristics.csv

In [20]:
# Cell 5 — Build song_characteristics.csv

from sklearn.preprocessing import MinMaxScaler

sc = merged[[
    'id', 'name', 'artists', 'album_name', 'genre', 'playlist_subgenre',
    'year', 'popularity', 'effective_popularity',
    'total_artist_followers', 'avg_artist_popularity'
] + AUDIO_FEATURES].copy()

sc = sc.rename(columns={'id': 'song_id'})

# ── Derived features ───────────────────────────────────────────────────────
# Danceability-energy composite
sc['dance_energy'] = (sc['danceability'] + sc['energy']) / 2

# Mood: positive high-energy vs calm
sc['mood_score'] = sc['valence'] * sc['energy']

# Normalised tempo bucket (slow / mid / fast)
sc['tempo_bucket'] = pd.cut(
    sc['tempo'], bins=[0, 90, 130, 300],
    labels=['slow', 'mid', 'fast']
).astype(str)

# Log-scale artist followers
sc['log_artist_followers'] = np.log1p(sc['total_artist_followers'])

# ── FIXED Popularity tier (with fallback) ───────────────────────────────────
try:
    sc['popularity_tier'] = pd.qcut(
        sc['effective_popularity'],
        q=5,
        labels=['very_low', 'low', 'mid', 'high', 'very_high']
    )
except ValueError:
    # fallback when duplicates collapse bins
    sc['popularity_tier'] = pd.qcut(
        sc['effective_popularity'],
        q=5,
        duplicates='drop'
    )

sc['popularity_tier'] = sc['popularity_tier'].astype(str)

# MinMax-scale audio features → sc_norm_* columns (useful for distance calcs)
scaler = MinMaxScaler()
normed = scaler.fit_transform(sc[AUDIO_FEATURES].fillna(0))
for i, col in enumerate(AUDIO_FEATURES):
    sc[f'norm_{col}'] = normed[:, i]

# ── Save ───────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'song_characteristics.csv'
sc.to_csv(out_path, index=False)
print(f'✅ song_characteristics.csv → {out_path}')
print(f'   Shape  : {sc.shape}')
print(f'   Columns: {sc.columns.tolist()}')

✅ song_characteristics.csv → data/large_hybrid/song_characteristics.csv
   Shape  : (30000, 34)
   Columns: ['song_id', 'name', 'artists', 'album_name', 'genre', 'playlist_subgenre', 'year', 'popularity', 'effective_popularity', 'total_artist_followers', 'avg_artist_popularity', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'dance_energy', 'mood_score', 'tempo_bucket', 'log_artist_followers', 'popularity_tier', 'norm_danceability', 'norm_energy', 'norm_loudness', 'norm_speechiness', 'norm_acousticness', 'norm_instrumentalness', 'norm_liveness', 'norm_valence', 'norm_tempo']


## Cell 6 — Build lyric_embeddings.csv (batched SentenceTransformer)

In [23]:
# Cell 6 — Build lyric_embeddings.csv
#
# Model : all-MiniLM-L6-v2  (384-dim, fast, M2-safe)
# Lyrics are truncated to 512 tokens at model level — no OOM risk.
# Batched encoding with show_progress_bar to track throughput.

from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'all-MiniLM-L6-v2'   # 384-dim; swap for larger if needed

print(f'Loading SentenceTransformer: {EMBED_MODEL} …')
model = SentenceTransformer(EMBED_MODEL)

# Prepare clean lyrics (truncated to 500 chars for speed; model handles the rest)
song_ids = merged['id'].tolist()
lyrics   = (
    merged['lyrics']
    .fillna('')
    .str.strip()
    .str[:500]          # hard char cap before tokenisation
    .tolist()
)

print(f'Encoding {len(lyrics):,} lyrics in batches of {EMBED_BATCH} …')

# encode() with batch_size handles chunking internally and is memory-safe
embeddings = model.encode(
    lyrics,
    batch_size=EMBED_BATCH,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # unit-norm → cosine sim = dot product
)

print(f'Embedding matrix: {embeddings.shape}')   # (N, 384)

# Build DataFrame
dim_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
emb_df   = pd.DataFrame(embeddings, columns=dim_cols)
emb_df.insert(0, 'song_id', song_ids)

# Save — float32 halves file size vs float64
emb_df[dim_cols] = emb_df[dim_cols].astype('float32')
out_path = OUT_DIR / 'lyric_embeddings.csv'
emb_df.to_csv(out_path, index=False)

print(f'✅ lyric_embeddings.csv → {out_path}')
print(f'   Shape: {emb_df.shape}  (~{out_path.stat().st_size / 1e6:.1f} MB)')

# Keep embedding matrix in memory for hard-negative mining (Cell 7)
# song_id → row index mapping
song_id_to_idx = {sid: i for i, sid in enumerate(song_ids)}
del lyrics  # free RAM
gc.collect()

Loading SentenceTransformer: all-MiniLM-L6-v2 …


README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 30,000 lyrics in batches of 128 …


Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Embedding matrix: (30000, 384)
✅ lyric_embeddings.csv → data/large_hybrid/lyric_embeddings.csv
   Shape: (30000, 385)  (~141.7 MB)


0

## Cell 7 — Pre-compute similarity index for hard-negative mining

In [27]:
# Cell 7 — Pre-compute similarity index for hard-negative mining
#
# We use a lightweight approximate approach:
#   1. Combine normalised audio features + lyric embedding (weighted)
#   2. Cluster with mini-batch K-Means → each song gets a cluster label
#   3. Hard negatives are drawn from same cluster OR same genre
# This avoids computing a full NxN similarity matrix (infeasible at 30k+).

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize

N_CLUSTERS = min(500, len(merged) // 20)   # ~20 songs/cluster on average

# Build combined feature matrix
audio_matrix = sc[[f'norm_{c}' for c in AUDIO_FEATURES]].values.astype('float32')

# Weight: 50% audio, 50% lyrics
AUDIO_WEIGHT  = 0.5
LYRICS_WEIGHT = 0.5

combined = np.hstack([
    audio_matrix  * AUDIO_WEIGHT,
    embeddings    * LYRICS_WEIGHT,
])
combined = normalize(combined, norm='l2')

print(f'Combined feature matrix: {combined.shape}')
print(f'Fitting MiniBatchKMeans with {N_CLUSTERS} clusters …')

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=4096,
    random_state=42,
    n_init=5,
    max_iter=100,
)
cluster_labels = kmeans.fit_predict(combined)

# Attach cluster to sc and merged
sc['cluster_id']     = cluster_labels
merged['cluster_id'] = cluster_labels

# Build inverted index: cluster_id → list of song row-indices
from collections import defaultdict
cluster_to_indices = defaultdict(list)
for idx, cid in enumerate(cluster_labels):
    cluster_to_indices[cid].append(idx)

# Build genre inverted index
genre_to_indices = defaultdict(list)
for idx, genre in enumerate(merged['genre'].values):
    genre_to_indices[genre].append(idx)

print(f'✅ Clusters ready. Avg cluster size: {len(merged) / N_CLUSTERS:.1f}')
del combined   # free RAM
gc.collect()

Combined feature matrix: (30000, 393)
Fitting MiniBatchKMeans with 500 clusters …
✅ Clusters ready. Avg cluster size: 60.0


0

## Cell 8 — Simulate users & generate interactions.csv

In [29]:
# Cell 8 — Simulate users & generate interactions.csv
#
# Each user gets:
#   • 1–3 primary genres (preference drawn from genre popularity)
#   • popularity bias  (mainstream ↔ niche spectrum)
#   • positives sampled proportionally by genre preference + popularity bias
#   • negatives = easy (random) + hard (same genre / same cluster)

from sklearn.preprocessing import minmax_scale

all_song_ids  = merged['id'].values
all_genres    = merged['genre'].values
all_clusters  = merged['cluster_id'].values
pop_scores    = merged['effective_popularity'].values.astype('float32')

# Normalise popularity → sampling probability
pop_normed = (pop_scores - pop_scores.min() + 1)
pop_normed = pop_normed / pop_normed.sum()

unique_genres = merged['genre'].value_counts()
genre_list    = unique_genres.index.tolist()
genre_probs   = (unique_genres.values / unique_genres.values.sum()).tolist()

rng = np.random.default_rng(42)


def sample_positives(preferred_genres: list, pop_bias: float, n: int) -> list:
    """
    Sample n positive song indices.
    pop_bias ∈ [0,1]: 0 → niche fan, 1 → mainstream listener.
    """
    # Build per-song score: genre_match * pop_component
    genre_match  = np.isin(all_genres, preferred_genres).astype('float32')
    # Blend genre preference with global popularity
    combined_w   = 0.7 * genre_match + 0.3 * (pop_bias * pop_normed)
    # Zero-out genres not preferred with small probability for diversity
    combined_w   = np.where(genre_match == 0, combined_w * 0.05, combined_w)
    if combined_w.sum() == 0:
        combined_w = pop_normed
    combined_w = combined_w / combined_w.sum()
    chosen = rng.choice(len(all_song_ids), size=min(n, len(all_song_ids)),
                        replace=False, p=combined_w)
    return chosen.tolist()


def sample_negatives(pos_idx: int, n_neg: int, hard_ratio: float,
                     positive_idx_set: set) -> list:
    """
    For one positive, sample n_neg negatives.
    hard_ratio fraction → same cluster (embed-similar) or same genre.
    """
    n_hard = max(1, int(n_neg * hard_ratio))
    n_easy = n_neg - n_hard
    negs   = []

    # ── Hard negatives (same cluster or same genre) ──────────────────
    cid   = int(all_clusters[pos_idx])
    genre = all_genres[pos_idx]

    hard_pool = list(
        set(cluster_to_indices[cid]) | set(genre_to_indices[genre])
    )
    hard_pool = [i for i in hard_pool if i != pos_idx and i not in positive_idx_set]

    if len(hard_pool) >= n_hard:
        hard_chosen = rng.choice(hard_pool, size=n_hard, replace=False).tolist()
    else:
        hard_chosen = hard_pool  # take all available
    negs.extend(hard_chosen)

    # ── Easy negatives (random, excluding positives & hards) ─────────
    n_easy_needed = n_neg - len(negs)
    if n_easy_needed > 0:
        exclude = positive_idx_set | set(negs) | {pos_idx}
        easy_pool = [i for i in range(len(all_song_ids)) if i not in exclude]
        # For speed on large datasets sample from shuffled pool
        rng.shuffle(easy_pool)
        negs.extend(easy_pool[:n_easy_needed])

    return negs[:n_neg]


# ── Main simulation loop ───────────────────────────────────────────────────
records = []   # list of dicts

for uid in tqdm(range(N_USERS), desc='Simulating users'):
    # User profile
    n_pref_genres = rng.integers(1, 4)   # 1–3 preferred genres
    pref_genres   = rng.choice(genre_list, size=int(n_pref_genres),
                               replace=False, p=genre_probs).tolist()
    pop_bias      = float(rng.uniform(0.2, 1.0))  # how mainstream the user is

    # Sample positives
    pos_indices = sample_positives(pref_genres, pop_bias, N_POSITIVES)
    pos_idx_set = set(pos_indices)

    for pos_idx in pos_indices:
        # Positive row
        records.append({
            'user_id'    : f'user_{uid:04d}',
            'song_id'    : all_song_ids[pos_idx],
            'label'      : 1,
            'is_hard_neg': False,
            'genre'      : all_genres[pos_idx],
        })

        # Negative rows
        neg_indices = sample_negatives(
            pos_idx, N_NEGATIVES, HARD_NEG_RATIO, pos_idx_set
        )
        for neg_idx in neg_indices:
            is_hard = (
                all_clusters[neg_idx] == all_clusters[pos_idx]
                or all_genres[neg_idx] == all_genres[pos_idx]
            )
            records.append({
                'user_id'    : f'user_{uid:04d}',
                'song_id'    : all_song_ids[neg_idx],
                'label'      : 0,
                'is_hard_neg': is_hard,
                'genre'      : all_genres[neg_idx],
            })

interactions_df = pd.DataFrame(records)

# ── Sanity stats ──────────────────────────────────────────────────────────
pos_count  = (interactions_df['label'] == 1).sum()
neg_count  = (interactions_df['label'] == 0).sum()
hard_count = ((interactions_df['label'] == 0) & interactions_df['is_hard_neg']).sum()
actual_hard_ratio = hard_count / neg_count if neg_count else 0

print(f'✅ interactions generated')
print(f'   Total rows     : {len(interactions_df):,}')
print(f'   Positives      : {pos_count:,}')
print(f'   Negatives      : {neg_count:,}')
print(f'   Hard negatives : {hard_count:,} ({actual_hard_ratio*100:.1f}%)')

out_path = OUT_DIR / 'interactions.csv'
interactions_df.to_csv(out_path, index=False)
print(f'   Saved → {out_path}')

Simulating users:   0%|          | 0/350 [00:00<?, ?it/s]

✅ interactions generated
   Total rows     : 147,000
   Positives      : 7,000
   Negatives      : 140,000
   Hard negatives : 75,006 (53.6%)
   Saved → data/large_hybrid/interactions.csv


## Cell 9 — Build user_features.csv

In [31]:
# Cell 9 — Build user_features.csv
#
# Per-user features:
#   genre_affinity_<G>     : fraction of positives in genre G  (affinity vector)
#   rel_pref_<audio>       : user mean audio feature vs global mean
#   listening_entropy      : Shannon entropy over genre distribution
#   diversity_score        : intra-list embedding diversity (mean pairwise cosine dist)
#   avg_popularity_pref    : mean effective_popularity of user positives
#   mainstream_score       : how mainstream the user is (0=niche, 1=mainstream)

from scipy.stats import entropy as scipy_entropy

# Attach audio features + embedding to interactions for agg
song_chars_indexed = sc.set_index('song_id')
emb_indexed        = emb_df.set_index('song_id')

global_audio_mean = song_chars_indexed[AUDIO_FEATURES].mean()
global_pop_mean   = song_chars_indexed['effective_popularity'].mean()

user_feature_rows = []

pos_interactions = interactions_df[interactions_df['label'] == 1].copy()

for user_id, group in tqdm(pos_interactions.groupby('user_id'),
                            desc='Building user features'):
    sids = group['song_id'].tolist()

    # ── Genre affinity vector ──────────────────────────────────────────
    genre_counts = group['genre'].value_counts()
    genre_fracs  = (genre_counts / genre_counts.sum()).to_dict()

    # ── Audio relative preference ──────────────────────────────────────
    valid_sids = [s for s in sids if s in song_chars_indexed.index]
    user_audio = song_chars_indexed.loc[valid_sids, AUDIO_FEATURES].mean()
    rel_pref   = (user_audio - global_audio_mean).to_dict()

    # ── Popularity preference ──────────────────────────────────────────
    avg_pop = song_chars_indexed.loc[valid_sids, 'effective_popularity'].mean()
    mainstream_score = float(avg_pop / 100.0) if not np.isnan(avg_pop) else 0.5

    # ── Listening entropy (genre diversity) ───────────────────────────
    genre_prob_vec = np.array(list(genre_fracs.values()))
    listen_entropy = float(scipy_entropy(genre_prob_vec + 1e-9))

    # ── Embedding diversity score ─────────────────────────────────────
    # Mean pairwise cosine distance among user positives
    valid_emb_sids = [s for s in sids if s in emb_indexed.index]
    if len(valid_emb_sids) >= 2:
        emb_mat = emb_indexed.loc[valid_emb_sids].values.astype('float32')  # (k, 384)
        # Dot product of unit-normed vectors = cosine similarity
        # Distance = 1 - cosine_sim
        sim_mat  = emb_mat @ emb_mat.T
        np.fill_diagonal(sim_mat, np.nan)
        diversity_score = float(np.nanmean(1.0 - sim_mat))
    else:
        diversity_score = 0.0

    # ── Assemble row ──────────────────────────────────────────────────
    row = {'user_id': user_id}

    # Genre affinity
    for g in genre_list:
        row[f'affinity_{g.replace(" ", "_").replace("/","_").replace("&","and")}'] = genre_fracs.get(g, 0.0)

    # Relative audio preference
    for feat, val in rel_pref.items():
        row[f'rel_pref_{feat}'] = float(val) if not np.isnan(val) else 0.0

    row['avg_popularity_pref'] = float(avg_pop) if not np.isnan(avg_pop) else global_pop_mean
    row['mainstream_score']    = mainstream_score
    row['listening_entropy']   = listen_entropy
    row['diversity_score']     = diversity_score
    row['n_positives']         = len(sids)

    user_feature_rows.append(row)


user_features_df = pd.DataFrame(user_feature_rows).fillna(0.0)

out_path = OUT_DIR / 'user_features.csv'
user_features_df.to_csv(out_path, index=False)

print(f'✅ user_features.csv → {out_path}')
print(f'   Shape: {user_features_df.shape}')
print(f'   Columns: {user_features_df.columns.tolist()[:10]} …')
print(f'   Entropy  mean: {user_features_df["listening_entropy"].mean():.3f}')
print(f'   Diversity mean: {user_features_df["diversity_score"].mean():.3f}')

Building user features:   0%|          | 0/350 [00:00<?, ?it/s]

✅ user_features.csv → data/large_hybrid/user_features.csv
   Shape: (350, 25)
   Columns: ['user_id', 'affinity_Rock', 'affinity_Pop', 'affinity_Electronic', 'affinity_Folk', 'affinity_Country', 'affinity_Hip-Hop', 'affinity_RandB', 'affinity_Jazz', 'affinity_Blues'] …
   Entropy  mean: 0.476
   Diversity mean: 0.746


## Cell 10 — Final validation & summary

In [34]:
# Cell 10 — Final validation & summary

import os

outputs = {
    'song_characteristics.csv': OUT_DIR / 'song_characteristics.csv',
    'lyric_embeddings.csv'    : OUT_DIR / 'lyric_embeddings.csv',
    'interactions.csv'        : OUT_DIR / 'interactions.csv',
    'user_features.csv'       : OUT_DIR / 'user_features.csv',
}

print('=' * 60)
print('PIPELINE COMPLETE — OUTPUT SUMMARY')
print('=' * 60)
all_ok = True
for name, path in outputs.items():
    if path.exists():
        df    = pd.read_csv(path, nrows=2)
        rows  = sum(1 for _ in open(path)) - 1   # fast line count
        size  = path.stat().st_size / 1e6
        print(f'  ✅ {name:<35} {rows:>8,} rows  {size:6.1f} MB  cols={len(df.columns)}')
    else:
        print(f'  ❌ MISSING: {name}')
        all_ok = False

# Integrity checks
print()
print('INTEGRITY CHECKS')

sc_ids   = set(pd.read_csv(OUT_DIR / 'song_characteristics.csv', usecols=['song_id'])['song_id'])
emb_ids  = set(pd.read_csv(OUT_DIR / 'lyric_embeddings.csv',    usecols=['song_id'])['song_id'])
int_ids  = set(pd.read_csv(OUT_DIR / 'interactions.csv',         usecols=['song_id'])['song_id'])

print(f'  song_characteristics ∩ lyric_embeddings coverage : {len(sc_ids & emb_ids) / len(sc_ids)*100:.1f}%')
print(f'  interactions song_ids ⊆ song_characteristics     : {int_ids.issubset(sc_ids)}')

int_df = pd.read_csv(OUT_DIR / 'interactions.csv')
pos  = (int_df['label']==1).sum()
neg  = (int_df['label']==0).sum()
hard = ((int_df['label']==0) & int_df['is_hard_neg']).sum()
print(f'  Positive : Negative ratio = 1 : {neg/pos:.1f}')
print(f'  Hard negative ratio        = {hard/neg*100:.1f}%  (target {HARD_NEG_RATIO*100:.0f}%)')

print()
if all_ok:
    print('All files generated successfully.')
    print()
    print('TRAINING COMMAND:')
    print()
    print(
        'python train.py '
        '--data-dir data/large_hybrid '
        '--epochs 10 '
        '--negatives-per-positive 20 '
        '--hard-negative-ratio 0.40 '
        '--lambda-rank 2.0 '
        '--lambda-point 0.5 '
        '--learning-rate 5e-4 '
        '--batch-size 512 '
        '--ranking-eval-candidates 200'
    )
else:
    print('⚠️  Some files are missing — re-run the failing cells.')

PIPELINE COMPLETE — OUTPUT SUMMARY
  ✅ song_characteristics.csv              30,000 rows    12.8 MB  cols=34
  ✅ lyric_embeddings.csv                  30,000 rows   141.7 MB  cols=385
  ✅ interactions.csv                     147,000 rows     6.8 MB  cols=5
  ✅ user_features.csv                        350 rows     0.1 MB  cols=25

INTEGRITY CHECKS
  song_characteristics ∩ lyric_embeddings coverage : 100.0%
  interactions song_ids ⊆ song_characteristics     : True
  Positive : Negative ratio = 1 : 20.0
  Hard negative ratio        = 53.6%  (target 40%)

All files generated successfully.

TRAINING COMMAND:

python train.py --data-dir data/large_hybrid --epochs 10 --negatives-per-positive 20 --hard-negative-ratio 0.40 --lambda-rank 2.0 --lambda-point 0.5 --learning-rate 5e-4 --batch-size 512 --ranking-eval-candidates 200


In [3]:
import torch
from pathlib import Path

MODEL_PATH = Path(
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens/artifacts/training/hybrid_model.pt"
)

print("Model exists:", MODEL_PATH.exists())

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

print(
    "Model size:",
    round(MODEL_PATH.stat().st_size / (1024 * 1024), 2),
    "MB"
)

ckpt = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\n" + "=" * 70)
print("CHECKPOINT KEYS")
print("=" * 70)
print(ckpt.keys())

print("\n" + "=" * 70)
print("MODEL CONFIG")
print("=" * 70)
print(ckpt["model_config"])

print("\n" + "=" * 70)
print("NUMBER OF SONGS")
print("=" * 70)
print(len(ckpt["index_to_song_id"]))

print("\nFirst 5 song IDs:")
print(ckpt["index_to_song_id"][:5])

if "index_to_user_id" in ckpt:
    print("\n" + "=" * 70)
    print("NUMBER OF USERS")
    print("=" * 70)
    print(len(ckpt["index_to_user_id"]))

Model exists: True
Model size: 9.45 MB

CHECKPOINT KEYS
dict_keys(['model_config', 'state_dict', 'training_metrics', 'data_artifacts', 'index_to_user_id', 'index_to_song_id', 'user_id_to_index', 'song_id_to_index'])

MODEL CONFIG
{'num_users': 350, 'num_songs': 30000, 'user_feature_dim': 24, 'metadata_dim': 67, 'lyric_embedding_dim': 384, 'latent_dim': 64, 'hidden_dim': 128, 'dropout': 0.1, 'interaction_type': 'dot'}

NUMBER OF SONGS
30000

First 5 song IDs:
['7FQyzz3MRRyIHB2xuPjQhL', '1bp2PcwZgR9LwBdoERsOuJ', '2AfFuqGQSTBI31kbqE1qo4', '5qTjvpAfdtyzauTprzgKao', '6V3SNkvi4BnfmZU0j7s9TQ']

NUMBER OF USERS
350
